# Solving MaxCut Problem Using Quantum Approximate Optimization Algorithm

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Solving MaxCut Problem Using Quantum Approximate Optimization Algorithm" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits


## Problem 1. Classical cost function

We can evaluate this classical function directly: iterate over all pairs of adjacent bits in the string and add $1$ to the sum if these bits are the same, leaving the sum unchanged (adding $0$ to it) if the bits are different.

In [ ]:
def classical_cost(x: list[int]) -> int:
    return sum([1 if x[j] == x[j + 1] else 0 for j in range(len(x) - 1)])

## Problem 2. Phase separation unitary

We can rewrite the classical cost as follows:

$$C(x) = \sum_{j = 0}^{N-2} C_j(x), \text{ where } C_j(x) = (x_j = x_{j+1}) = x_j x_{j+1} + (1-x_j)(1-x_{j+1})$$

The effect of the phase separation unitary on a basis state is defined as follows:

$$U_C(\gamma)|x\rangle = e^{-i\gamma C(x)}|x\rangle$$

Since all the terms $C_j(x)$ in $C(x)$ are scalars, not operators, we can rewrite the exponent of a sum as a product of exponents:

$$e^{-i\gamma C(x)} = e^{-i\gamma \sum C_j(x)} = \prod e^{-i\gamma C_j(x)}$$

Now, we can apply each relative phase $e^{-i\gamma C_j(x)}$ separately. This relative phase only depends on two qubits, $x_j$ and $x_{j+1}$, and can be further split in two terms:

$$e^{-i\gamma C_j(x)} = e^{-i \gamma (x_j x_{j+1} + (1-x_j)(1-x_{j+1}) )} = e^{-i \gamma x_j x_{j+1}} e^{-i \gamma (1-x_j)(1-x_{j+1})}$$

We know that $x_j$ and $x_{j+1}$ can each be $0$ or $1$. This means that each term is not $1$ only for one combination of $x_j$ and $x_{j+1}$ values:

$$e^{-i \gamma x_j x_{j+1}} = \begin{cases} e^{-i \gamma}, x_j = x_{j+1} = 1 \\ 1 \text{ otherwise}\end{cases}$$

$$e^{-i \gamma (1-x_j)(1-x_{j+1})} = \begin{cases} e^{-i \gamma}, x_j = x_{j+1} = 0 \\ 1 \text{ otherwise}\end{cases}$$

We can implement each of these terms as a controlled phase gate, applying a relative phase $e^{-i \gamma}$ to the basis states $\ket{00}$ and $\ket{11}$, respectively.

In [ ]:
def phase_separation_unitary(reg: Qubits, gamma: float) -> None:
    from psiqdk.workbench import units
    for j in range(len(reg) - 1):
        reg[j].phase(-gamma * units.rad, cond=reg[j + 1])
        reg[j].x()
        reg[j].phase(-gamma * units.rad, cond=~reg[j + 1])
        reg[j].x()

## Problem 3. The mixer unitary

The mixer unitary can be represented as a tensor product of unitaries acting on individual qubits, since each term $X_j$ acts on a different qubit:

$$U_B(\beta) \ket{x} = e^{-i\beta \sum_{j=0}^{N-1} X_j} \ket{x} = \prod_{j=0}^{N-1} e^{-i\beta X_j} \ket{x} = \bigotimes_{j=0}^{N-1} e^{-i\beta X_j} \ket{x_j}$$

Conveniently, we have access to a built-in gate that implements the unitary $e^{-i\beta X_j}$: it's $R_x(2\beta)$!

We need to apply the same $R_x$ gate with the same parameters to all qubits. In Workbench, you can compress the for loop for this into a single multi-target gate call.

In [ ]:
def mixer_unitary(reg: Qubits, beta: float) -> None:
    from psiqdk.workbench import units
    reg.rx(2 * beta * units.rad)

> Copyright (c) 2026 PsiQuantum